### Initalization

In [ ]:
%load_ext autoreload
%autoreload 2

%reload_ext autoreload

In [ ]:
##====================================================================================================================================##
## PARAMETERS AND IMPORTS

from satellite_RFI.src import tools
from satellite_RFI.src.satellite_sims import satellite_sim as ss
from satellite_RFI.src import attenuation_function as af

import natsort   # https://pypi.org/project/natsort/
# https://stackoverflow.com/questions/4836710/is-there-a-built-in-function-for-string-natural-sort

import sys
sys.path.insert(0, '../../param_import/')
from imports import *
import param_multi_mask as pm

from matplotlib.ticker import MaxNLocator

##====================================================================================================================================##
## Saving images
savegig = False

In [ ]:
# blocks = ['1551037708', '1551055211', '1553966342', '1554156377', '1556052116', '1556138397', '1562857793']
# folders = ['test_results_sat_3', 'test_results_sat_12', 'test_results_sat_3', 'test_results_sat_3', 'test_results_sat_3', 'test_results_sat_3', 'test_results_sat_3']

blocks = ['1551055211']
folders = ['test_results_sat_12']
sat_cat = '../../Satellite_Catalogue/satellite_constellation_catalog.csv'

In [ ]:
def katdal_information(data_save):
    '''
    Function for looking up the katdal information
    '''
    
    if sys.version_info.major==2:
            katdal_info = pickle.load(open(data_save + '_katdal_info.p', 'rb'))

    elif sys.version_info.major==3:
        katdal_info = pickle.load(open(data_save + '_katdal_info.p', 'rb'), encoding = 'latin1')
            
    info = [katdal_info[i] for i in katdal_info.keys()]
    nd_s0 = katdal_info['nd_s0']
    nd_s0_coords = katdal_info['nd_s0_coords']
    nd_s0_coords2 = katdal_info['nd_s0_coords2']
    nd_s0_pos = katdal_info['nd_s0_pos']
    frequency = katdal_info['frequency']
    
    return nd_s0, nd_s0_coords, nd_s0_coords2, nd_s0_pos, frequency

### Chi Square Method

In [ ]:
##CHI SQAURE METHOD
def chisq_func2(
    block_id, 
    a_param, 
    s_param = None, 
    damper = None, 
    satellites_only = False, 
    sat_info = pm.satellite_catalogue, 
    add_sub = [
        True, 
        False
    ],
    frequency_slice = False, 
    time_slice = False, 
    t_mask = False, 
    d_mask = False,
    pt_mask = False,
    time_avg = False, 
    chi_sigma = False, 
    verbose = False
):
    
    data_save = '/idia/projects/hi_im/satellite_rfi/Testing/' + block_id + '/'
    nd_s0, nd_s0_coords, nd_s0_coords2, nd_s0_pos, frequency = katdal_information(
        data_save = data_save + block_id
    )

    if d_mask == False:
        satellites = None
    else:
        satellites = data_save + 'nearby_satellites/nearby_satellite_close_angle_' + d_mask + '.p'
    
    """INITIALIZING THE SATELLITE FUNCTION"""
    sat = ss(
        file_name = block_id, 
        sats_only = satellites_only, 
        data_loc = data_save, 
        sat_loc = data_save,
        survey_info = [nd_s0, 
                  nd_s0_coords, 
                  frequency], 
        sat_info = sat_info,
        plots_loc = data_save,
        sat_beam = pm.beam_model + '_beam_' + str(pm.fs) + '_' + str(pm.fe) + 'MHz', 
        frequency_range = [pm.fs, 
                         pm.fe], 
        constellations = pm.constellations_remain,
        nearby_satellites = satellites,
        verbose = False
    )
    
    """EXCECUTING THE THE SATELLITE SIM FUNCTION"""
    sat.excecute(
        a_param = a_param, 
        obs_time_start = time_slice[0], 
        obs_time_end = time_slice[1], 
        obs_frequency_start = frequency_slice[0], 
        obs_frequency_end = frequency_slice[1], 
        file_bias_choice = pm.bias, 
        add_sub = add_sub, 
        attenuation_func = damper, 
        attenuation_sigma = s_param, 
        bandsize = None,
        verbose = False
    )

# ========================================================================================#    
# FREQUENCY SLICE
    frange_slice = sat.frequency_band[
        sat.frequency_idx[0]:sat.frequency_idx[1]
    ]  
# ========================================================================================#    
# MASKING
# Thermal Masking 
    if (
        t_mask != False 
        and d_mask == False
        and pt_mask == False
    ):
        print ("Thermal mask: " + str(t_mask) + " Kelvin")
        zero_arr = np.zeros(sat.calibration_data_slice.shape)
        mask_idx = np.where(sat.calibration_data_slice > t_mask)

        zero_arr[mask_idx]=1
        
        simulation = np.ma.array(
            data = sat.simulation_TOD_slice.T, 
            mask = zero_arr.T
        )    
        
        data = np.ma.array(
            data = sat.calibration_data_slice.T, 
            mask = zero_arr.T
        )  #DATA

# Angular Masking
    elif (
        d_mask != False 
        and t_mask == False
        and pt_mask == False
    ):
        print ("Angular mask: " + str(d_mask) + " degrees")
        simulation = np.ma.array(
            data=sat.simulation_TOD_slice.T, 
            mask=sat.mask_nearby_satellites_slice
        )
        
        data = np.ma.array(
            data = sat.calibration_data_slice.T, 
            mask = sat.mask_nearby_satellites_slice
        )

# Pixel Timeline Masking
    elif (
        d_mask == False 
        and t_mask == False
        and pt_mask != False
    ):
        print ("Pixel timeline mask: " + str(pt_mask) + " Kelvin.")
        masker_tod = tools.time_line_masker(
            data_in = sat.calibration_data_slice.T, 
            t_value = pt_mask
        )

        simulation = np.ma.array(
            data = sat.simulation_TOD_slice.T, 
            mask = masker_tod
        )
        
        data = np.ma.array(
            data = sat.calibration_data_slice.T, 
            mask = masker_tod
        )
        
# No Masking Applied
    else:
        print ("No masking applied")
        
        simulation = np.ma.array(
            data = sat.simulation_TOD_slice.T, 
            mask = None)
        
        data = np.ma.array(
            data = sat.calibration_data_slice.T, 
            mask = None)
        
# Time Avering
    if time_avg != False:
        print ("Time averageing for every " + str(time_avg) + " seconds")
        #Averaging over time - MISSING TIMER PARAMETER
        data = tools.waterfall_avg_time(
            timer = nd_s0[sat.time_idx[0]:sat.time_idx[1]], 
            size = time_avg, waterfall = data)
        
        simulation = tools.waterfall_avg_time(
            timer = nd_s0[sat.time_idx[0]:sat.time_idx[1]],
            size = time_avg, waterfall = simulation)
        
        time_idx = sat.time_idx


    
    """CHI SQAURE NUMERATOR"""
    chi_num = simulation - data  # WANT THIS VALUE

    """CHI^2 SIGMA CHOICE"""
    if chi_sigma == True:
        sig = data
        # sig=tools.radiometer_eq(data=data, n_dish=58)  
        # Note this is sigma (expected noise level), it must be squared to give the radiometer equation
    else: 
        sig = 1
    
    """CHI SQUARE VALUE"""
    chi_sq = np.ma.sum(chi_num**2 / sig**2) 
      
    chi_sq_N = chi_sq / np.ma.size(data)
    
    if verbose == True:
        print ("Chi numerator value: " + str(np.ma.sum(chi_num)))
        print ("Chi Square value: " + str(chi_sq))
        print ("Chi Square N value: " + str(chi_sq_N))
    
    if time_avg != False: 
        return data, simulation, chi_sq_N, frange_slice, sat.time_idx#, sat.sat_data_adjusted
    
    else:
        return data, simulation, chi_sq_N, frange_slice#, sat.sat_data_adjusted

## ORIGINAL IN PAPER

### No-Masking Simulation

In [ ]:
def no_mask_alpha(block_id, folder_id):
    '''
    Function to return the fractional and residual alpha values for the no mask scenario
    '''
    
    loc = '/idia/projects/hi_im/satellite_rfi/Testing/' + block_id + '/' + folder_id + '/no_mask/'
    file_list = os.listdir(loc)
    file_list.sort()
    
    frac = pickle.load(open(loc + file_list[0], 'rb'))
    resi = pickle.load(open(loc + file_list[1], 'rb'))
    
    return frac, resi

def no_mask_alpha_plot(frac, resi, block_id):
    '''
    Function to plot out the alpha values for the no mask scenario
    '''
    fig, ax = plt.subplots(figsize=(10, 4), nrows=1, ncols=1)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xticks(range(1, 21 + 1))

    ax.set_title(r'Block: '+block_id+' No-Mask')
    ax.plot(np.arange(1,22), frac['best-fit'], 'o', label=r'$\sigma_{D}=C_1$')
    ax.plot(np.arange(1,22), resi['best-fit'], '*', label=r'$\sigma_D=C_2$')
    ax.axhline(0, color='black', linestyle='--')
    ax.set_xlabel(r'Signal Position $[\alpha_{i}]$')
    ax.set_ylabel('Amplitude')
    ax.legend()
    fig.tight_layout()
    if savegig==True:
        fig.savefig('./nm_fitting_alpha.pdf')
    fig.show()
    
def no_mask_simulation_1d_plot(frac, resi, frequency, block_id):
    '''
    Function for plotting the compariosn between the data and simulation for 1d
    '''
    fig, axs = plt.subplots(figsize=(20, 4), ncols=2, nrows=1, sharex=True, sharey=True)
    fig.suptitle(r'Block: '+block_id+' No-Mask')



    ax=axs[0]
    ax.plot(frequency, np.mean(frac[0], axis=0), label='Observation')
    ax.plot(frequency, np.mean(frac[1], axis=0), '--', label='Simulation')
    ax.set_ylabel('Temperature [K]')
    ax.set_xlabel('Frequency [MHz]')
    textstr = '\n'.join((
        r'$\sigma_D=C_1$',
        r'FoM$=%.5f$' % (np.round(mno_sf[2], 5), ),))

    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax.text(0.015, 0.95, textstr, transform=ax.transAxes, fontsize=18,
            verticalalignment='top', bbox=props)

    ax=axs[1]
    ax.plot(frequency, np.mean(resi[0], axis=0), label='Observation')
    ax.plot(frequency, np.mean(resi[1], axis=0), '--', label='Simulation')
    ax.set_xlabel('Frequency [MHz]')
    ax.set_ylabel('Temperature [K]')
    textstr = '\n'.join((
        r'$\sigma_D=C_2$',
        r'FoM$=%.2f$' % (np.round(mno_sr[2], 2), ),))

    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax.text(0.015, 0.95, textstr, transform=ax.transAxes, fontsize=18,
            verticalalignment='top', bbox=props)
    ax.legend()
    fig.tight_layout()
    if savegig==True:
        fig.savefig('./no_mask_best_fit_1d.pdf')
    
    fig.show()

def no_mask_simulation_2d_plot(plot_info, frequency, block_id):
    '''
    Function to plot the 2d no mask scenario plot
    '''
    plots_title = ['Observation', r'$C_1$', r'$C_2$']
    fig, axs = plt.subplots(figsize=(12, 4), nrows=1, ncols=3, sharey=True)
    fig.suptitle(r'Block: '+block_id+' No-Mask')  
    for i in range(len(plot_info)):
            ax = axs[i]

            cax = ax.imshow(plot_info[i], aspect='auto', extent=[frequency[0], frequency[-1], pm.nd_s0[-1], pm.nd_s0[0]], vmax=np.max(mno_sr[0]), vmin=np.min(mno_sr[0]))
            cbar = fig.colorbar(cax, ax=ax)
            cbar.set_label(r'Temperature [K]', rotation=270, labelpad=20, y=0.45)

            ax.set_title(plots_title[i])
            ax.set_ylabel('Time [sec]')
            ax.set_xlabel('Frequency [MHz]')

    fig.tight_layout()
    if savegig==True:
        fig.savefig('./no_mask_best_fit_2d.pdf')
        
    fig.show()








In [ ]:
no_mask_dic = {}
for bi in range(len(blocks)):
    # Alpha values
    frac, resi = no_mask_alpha(
        block_id = blocks[bi], 
        folder_id = folders[bi]
    )
    
    # Adding a dictionary term to store alpha information
    no_mask_dic.update({blocks[bi] : [frac, resi]})
    
    no_mask_alpha_plot(
        frac = frac, 
        resi = resi, 
        block_id = blocks[bi]
    )
    # Simulation
    mno_sf = chisq_func2(
        block_id = blocks[bi],
        a_param = frac['best-fit'],
        s_param = None,
        damper = None, 
        sat_info = sat_cat,
        frequency_slice = [
            1100, 
            1350
        ],
        time_slice = [
            None, 
            None
        ], 
        t_mask = False,
        d_mask = False, 
        time_avg = False, 
        chi_sigma = True, 
        verbose = False
    )

    mno_sr = chisq_func2(
        block_id = blocks[bi],
        a_param = resi['best-fit'],
        s_param = None,
        damper = None, 
        sat_info = sat_cat,
        frequency_slice = [
            1100, 
            1350
        ],
        time_slice = [
            None, 
            None
        ], 
        t_mask = False,
        d_mask = False, 
        time_avg = False, 
        chi_sigma = False,
        verbose = False)

    f_slice = mno_sr[3]
    no_mask_simulation_1d_plot(
        frac = mno_sf, 
        resi = mno_sr, 
        frequency = f_slice, 
        block_id = blocks[bi]
    )
    
    no_mask_simulation_2d_plot(
        plot_info = [
            mno_sr[0], 
            mno_sf[1], 
            mno_sr[1]
        ],
        frequency = f_slice, 
        block_id = blocks[bi]
    )



### Masking - Angular Simulation

In [ ]:
def angular_masking_alpha(block_id, folder_id):
    loc = '/idia/projects/hi_im/satellite_rfi/Testing/'+block_id+'/'+folder_id+'/degree_mask/'

    file_list = os.listdir(loc)
    file_list = natsort.natsorted(file_list, key=lambda y: y.lower())    

    idx = np.arange(len(file_list))
    even_idx = idx[::2]
    odd_idx = idx[1::2]
    
    frac = [pickle.load(open(loc+file_list[di],'rb'))
               for di in even_idx]

    resi = [pickle.load(open(loc+file_list[di],'rb'))
               for di in odd_idx]
    
    return frac, resi


def angular_masking_alpha_plot(frac, resi, block_id, degree, deg_name):
    '''
    Angular masking plot for the degree alpha 
    '''
    if len(degree)==2:
        ls = ['o', '*']
    if len(degree)==5:
        ls = ['o', '*', 'x', '.', 'o']
    fig, axs = plt.subplots(figsize=(10, 8), nrows=2, ncols=1, sharey=True)
    fig.suptitle(r'Block: '+block_id+' Angular mask')
    
    ax=axs[0]
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xticks(range(1, 21 + 1))
    ax.set_ylabel('Amplitude')
    textstr = '\n'.join((
        r'$\sigma_D=C_1$', ),)
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax.text(0.06, 0.95, textstr, transform=ax.transAxes, fontsize=14,
            verticalalignment='top', bbox=props)

    for di in range(len(frac)):
        ax.plot(np.arange(1,22), frac[di]['best-fit'], ls[di], label=r'Degree: '+deg_name[di])
    ax.axhline(0, color='black', linestyle='--')
    ax.legend()

    
    ax=axs[1]
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xticks(range(1, 21 + 1))
    ax.set_xlabel(r'Signal Position $[\alpha_{i}]$')
    ax.set_ylabel('Amplitude')
    textstr = '\n'.join((
        r'$\sigma_D=C_2$', ),)
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax.text(0.06, 0.95, textstr, transform=ax.transAxes, fontsize=18,
            verticalalignment='top', bbox=props)
    for di in range(len(frac)):
        ax.plot(np.arange(1,22), resi[di]['best-fit'], ls[di], label=r'Degree: '+deg_name[di])
    ax.axhline(0, color='black', linestyle='--')
    fig.tight_layout()
    
    if savegig==True:
        fig.savefig('./deg_fitting_alpha.pdf')
        
    fig.show()
    
    
def angular_masking_1d_simulation(frac, resi, block_id, degree, degree_name):
    '''
    Function that plots the 1d simulation results for the angular masking scenario
    '''

    fig, axs = plt.subplots(figsize=(20, 3*len(degree)), nrows=len(degree), ncols=2, sharey=True)
    fig.suptitle(r'Block: '+block_id+' Angular Masking')
    for ri in range(len(degree)):
        ax=axs[ri,0]
        ax.plot(frac[ri][3], np.mean(frac[ri][0], axis=0), label='Observation')
        ax.plot(frac[ri][3], np.mean(frac[ri][1], axis=0), '--', label='Simulation')
        textstr = '\n'.join((
            r'$\sigma_D=C_1$',
            r'Degree='+degree_name[ri],
            r'FoM$=%.5f$' % (np.round(frac[ri][2], 5), ),))

        props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
        ax.text(0.015, 0.95, textstr, transform=ax.transAxes, fontsize=18,
                verticalalignment='top', bbox=props)
        if ri==1:
            ax.set_xlabel('Frequency [MHz]')
        ax.set_ylabel('Temperature [K]')

        ax=axs[ri,1]
        ax.plot(resi[ri][3], np.mean(resi[ri][0], axis=0), label='Observation')
        ax.plot(resi[ri][3], np.mean(resi[ri][1], axis=0), '--', label='Simulation')
        textstr = '\n'.join((
            r'$\sigma_D=C_2$',
            r'Degree='+degree_name[ri],
            r'FoM$=%.2f$' % (np.round(resi[ri][2], 3), ),))
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
        ax.text(0.015, 0.95, textstr, transform=ax.transAxes, fontsize=18,
                verticalalignment='top', bbox=props)
        if ri==1:
            ax.set_xlabel('Frequency [MHz]')

            ax.legend(loc='upper right')
    
    fig.tight_layout()
    fig.show()
                            
def angular_masking_2d_simulation(frac, resi, block_id, degree, degree_name):
    '''
    Function for plotting the 2d information of the observation vs the model in fractional and residual
    '''
    
    plots_title = ['Observation', r'$C_1$', r'$C_2$']
    fig, axs = plt.subplots(figsize=(16, 3*len(degree)), nrows=len(degree), ncols=3, sharey=True, sharex=True)
    fig.suptitle(r'Block: '+block_id+' Angular mask')
    for ri in range(len(degree)):
        for ci in range(3):
                xi = 3*ri+ci
                ax = axs[ri, ci]
                if ri==0:
                    ax.set_title(plots_title[ci])

                if ci==0:
                    cax = ax.imshow(resi[ri][0], aspect='auto', extent=[resi[ri][3][0], resi[ri][3][-1], pm.nd_s0[-1], pm.nd_s0[0]], vmax=np.max(resi[ri][0]), vmin=np.min(resi[ri][0]))
                elif ci==1:
                    cax = ax.imshow(frac[ri][1], aspect='auto', extent=[resi[ri][3][0], resi[ri][3][-1], pm.nd_s0[-1], pm.nd_s0[0]], vmax=np.max(resi[ri][0]), vmin=np.min(resi[ri][0]))
                else:
                    cax = ax.imshow(resi[ri][1], aspect='auto', extent=[resi[ri][3][0], resi[ri][3][-1], pm.nd_s0[-1], pm.nd_s0[0]], vmax=np.max(resi[ri][0]), vmin=np.min(resi[ri][0]))  

                cbar = fig.colorbar(cax, ax=ax)
                cbar.set_label(r'Temperature [K]', rotation=270, labelpad=20, y=0.45)

                if ci==0:
                    ax.set_ylabel('Time [sec]')
                if ri==1:
                    ax.set_xlabel('Frequency [MHz]')

    fig.tight_layout()
    if savegig==True:
        fig.savefig('./deg_fitting_waterfall.pdf')
        
    fig.show()


In [ ]:
angular_mask_dic = {}
for bi in range(len(blocks)):
    
    if blocks[bi] == '1551055211' or blocks[bi] == '1554156377' or blocks[bi] == '1556052116' or blocks[bi] == '1556138397' or blocks[bi] == '1562857793':
        degree = ['1F', '5F']
        degree_name = [r'$1^{\circ}$F', r'$5^{\circ}$F']
    
    if blocks[bi] == '1553966342' or blocks[bi] == '1551037708':
        degree = ['1F', '2F', '3F', '4F', '5F']
        degree_name = [r'$1^{\circ}$F', r'$2^{\circ}$F', r'$3^{\circ}$F', r'$4^{\circ}$F', r'$5^{\circ}$F']
    
    
    
    # Alpha parameters
    frac, resi = angular_masking_alpha(
        block_id = blocks[bi],
        folder_id = folders[bi]
    )
    
    angular_mask_dic.update({blocks[bi] : [frac, resi]})
    angular_masking_alpha_plot(
        frac = frac, 
        resi = resi, 
        block_id = blocks[bi], 
        degree = degree, 
        deg_name = degree_name
    )
    
    # Simulation
    md_sf = [
        chisq_func2(
            block_id = blocks[bi], 
            a_param = frac[di]['best-fit'],
            s_param = None, 
            damper = None, 
            sat_info = sat_cat, 
            frequency_slice = [1100, 1350],
            time_slice = [None, None], 
            t_mask = False,  
            d_mask = deg, 
            time_avg = False, 
            chi_sigma = True, 
            verbose = False
            ) for di, deg in enumerate(degree)
    ]

    md_sr = [
        chisq_func2(
            block_id = blocks[bi], 
            a_param = resi[di]['best-fit'],
            s_param = None, 
            damper = None, 
            sat_info = sat_cat, 
            frequency_slice = [1100, 1350],
            time_slice = [None, None], 
            t_mask = False,  
            d_mask = deg, 
            time_avg = False, 
            chi_sigma = False,
            verbose = False
        ) for di, deg in enumerate(degree)
    ]

    
    angular_masking_1d_simulation(
        frac = md_sf, 
        resi = md_sr, 
        block_id = blocks[bi],
        degree = degree, 
        degree_name = degree_name
    )
    
    angular_masking_2d_simulation(
        frac = md_sf, 
        resi = md_sr, 
        block_id = blocks[bi],
        degree = degree, 
        degree_name = degree_name
    )
    


### Thermal-masking

In [ ]:


def thermal_masking_alpha(block_id, folder_id):
    loc = '/idia/projects/hi_im/satellite_rfi/Testing/' + block_id + '/' + folder_id + '/thermal_mask/'
 
    file_list = os.listdir(loc)
    natsort.natsorted(file_list, key = lambda y: y.lower()) 
    
    idx = np.arange(len(file_list))
    even_idx = idx[::2]
    odd_idx = idx[1::2]
    
    frac = [
        pickle.load(
            open(
                loc + file_list[di], 'rb'))
               for di in even_idx
    ]

    resi = [
        pickle.load(
            open(
                loc + file_list[di], 'rb'))
               for di in odd_idx
    ]
    
    return frac, resi

def thermal_masking_alpha_plot(frac, resi, block_id, temperature):
    '''
    Thermal masking plot for the temperature alpha 
    '''
    
    ls = ['o', '*', 'x']

    fig, axs = plt.subplots(figsize=(10, 8), nrows=2, ncols=1, sharey=True)

    fig.suptitle(r'Block: '+block_id+' Thermal mask')

    ax=axs[0]
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xticks(range(1, 21 + 1))
    ax.set_ylabel('Amplitude')

    textstr = '\n'.join((
        r'$\sigma_D=C_1$', ),)

    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax.text(0.06, 0.95, textstr, transform=ax.transAxes, fontsize=18,
            verticalalignment='top', bbox=props)


    for ti in range(len(frac)):
        ax.plot(np.arange(1,22), frac[ti]['best-fit'], ls[ti], label=r'Temperature: '+temperature[ti]+' K')
    ax.axhline(0, color='black', linestyle='--')


    ax.legend()

    ax=axs[1]
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xticks(range(1, 21 + 1))
    ax.set_xlabel(r'Signal Position $[\alpha_{i}]$')
    ax.set_ylabel('Amplitude')

    textstr = '\n'.join((
        r'$\sigma_D=C_2$', ),)

    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax.text(0.06, 0.95, textstr, transform=ax.transAxes, fontsize=18,
            verticalalignment='top', bbox=props)

    for ti in range(len(resi)):
        ax.plot(np.arange(1,22), resi[ti]['best-fit'], ls[ti], label=r'Temperature: '+temperature[ti]+' K')
    ax.axhline(0, color='black', linestyle='--')


    fig.tight_layout()
    if savegig==True:
        fig.savefig('./thermal2_fitting_alpha.pdf')
        
    fig.show()
    
    
def thermal_masking_1d_simulation(frac, resi, block_id, temperature):
    '''
    Function for determining the 1d comparison between the simulation and the model
    '''
    fig, axs = plt.subplots(figsize=(20, 3*len(temperature)), nrows=len(temperature), ncols=2, sharey=False)

    fig.suptitle('Block:'+block_id+' Thermal masking')
    for ri in range(len(temperature)):
        ax=axs[ri,0]
        ax.plot(frac[ri][3], np.mean(frac[ri][0], axis=0), label='Observation')
        ax.plot(frac[ri][3], np.mean(frac[ri][1], axis=0), '--', label='Simulation')
        textstr = '\n'.join((
            r'$\sigma_{D}=C_1$',
            r'Thermal='+temperature[ri]+' K',
            r'FoM$=%.5f$' % (np.round(frac[ri][2], 5), ),))

        props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
        ax.text(0.015, 0.95, textstr, transform=ax.transAxes, fontsize=18,
                verticalalignment='top', bbox=props)
        if ri==2:
            ax.set_xlabel('Frequency [MHz]')
        ax.set_ylabel('Temperature [K]')
        if ri==0:
            ax.legend()
            
        ax=axs[ri,1]
        ax.plot(resi[ri][3], np.mean(resi[ri][0], axis=0), label='Observation')
        ax.plot(resi[ri][3], np.mean(resi[ri][1], axis=0), '--', label='Simulation')
        textstr = '\n'.join((
            r'$\sigma_{D}=C_2$',
            r'Thermal='+temperature[ri]+' K',
            r'FoM$=%.2f$' % (np.round(resi[ri][2], 2), ),))

        props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
        ax.text(0.015, 0.95, textstr, transform=ax.transAxes, fontsize=18,
                verticalalignment='top', bbox=props)

        if ri==2:
            ax.set_xlabel('Frequency [MHz]')



    fig.tight_layout()
    if savegig==True:
        fig.savefig('./thermal2_fitting.pdf')
        
    fig.show()
    
def thermal_masking_2d_simulation(frac, resi, block_id, temperature):
    plots_title = ['Observation', r'$C_1$', r'$C_2$']
    fig, axs = plt.subplots(figsize=(16, 8), nrows=3, ncols=3, sharey=True, sharex=True)
    fig.suptitle('Block:'+block_id+' Thermal masking')
    for ri in range(len(temperature)):
        for ci in range(3):
                xi = 3*ri+ci
                ax = axs[ri, ci]
                if ri==0:
                    ax.set_title(plots_title[ci])

                if ci==0:
                    cax = ax.imshow(resi[ri][0], aspect='auto', extent=[resi[ri][3][0], resi[ri][3][-1], pm.nd_s0[-1], pm.nd_s0[0]], vmax=np.max(mt_sr[ri][0]), vmin=np.min(mt_sr[ri][0]))
                elif ci==1:
                    cax = ax.imshow(resi[ri][1], aspect='auto', extent=[resi[ri][3][0], resi[ri][3][-1], pm.nd_s0[-1], pm.nd_s0[0]], vmax=np.max(mt_sr[ri][0]), vmin=np.min(mt_sr[ri][0]))
                else:
                    cax = ax.imshow(frac[ri][1], aspect='auto', extent=[resi[ri][3][0], resi[ri][3][-1], pm.nd_s0[-1], pm.nd_s0[0]], vmax=np.max(mt_sr[ri][0]), vmin=np.min(mt_sr[ri][0]))  

                cbar = fig.colorbar(cax, ax=ax)
                cbar.set_label(r'Temperature [K]', rotation=270, labelpad=20, y=0.45)

                if ci==0:
                    ax.set_ylabel('Time [sec]')
                if ri==2:
                    ax.set_xlabel('Frequency [MHz]')

    fig.tight_layout()
    if savegig==True:
        fig.savefig('./thermal_fitting_waterfall.pdf')



In [ ]:
temperature = ['25', '50', '100']
thermal_masking_dic = {}
for bi in range(len(blocks)):
    # Alpha parameters
    frac, resi = thermal_masking_alpha(
        block_id = blocks[bi], 
        folder_id = folders[bi]
    )
    thermal_masking_dic.update({blocks[bi] : [frac, resi]})
    
    thermal_masking_alpha_plot(
        frac = frac, 
        resi = resi, block_id = blocks[bi],
        temperature = temperature
    )
    
    
    #Simulation
    mt_sf = [chisq_func2(
        block_id = blocks[bi],
        a_param = frac[ti]['best-fit'],
        s_param = None, 
        damper =None, 
        frequency_slice = [
            1100, 
            1350
        ], 
        time_slice = [
            None,
            None
        ], 
        t_mask = int(temp), 
        d_mask = False, 
        time_avg = False, 
        chi_sigma = True, 
        verbose = False
    ) 
             for ti,temp in enumerate(temperature)
            ]
  
    mt_sr = [chisq_func2(
        block_id = blocks[bi],
        a_param = resi[ti]['best-fit'],
        s_param = None,
        damper = None, 
        frequency_slice = [
            1100, 
            1350
        ], 
        time_slice = [
            None, 
            None
        ], 
        t_mask = int(temp), 
        d_mask = False, 
        time_avg = False, 
        chi_sigma = False, 
        verbose = False
    ) 
             for ti,temp in enumerate(temperature)
            ]
    
    thermal_masking_1d_simulation(
        frac = mt_sf,
        resi = mt_sr,
        block_id = blocks[bi],
        temperature = temperature
    )
    
    thermal_masking_2d_simulation(
        frac = mt_sf,
        resi = mt_sr,
        block_id = blocks[bi],
        temperature = temperature
    )
    



### Pixel timeline-masking

In [ ]:
def pixel_timeline_alpha(block_id, folder_id):
    '''
    Function to return the fractional and residual alpha values for the pixel timeline mask scenario
    '''
    loc = '/idia/projects/hi_im/satellite_rfi/Testing/' + block_id + '/' + folder_id + '/pixel_timeline_mask/'
    file_list = os.listdir(loc)
    file_list = natsort.natsorted(file_list, key=lambda y: y.lower()) 
    idx = np.arange(len(file_list))
    even_idx = idx[::2]
    odd_idx = idx[1::2]
    
    frac = [
        pickle.load(
        open(
            loc + file_list[di], 'rb'))
               for di in even_idx
           ]
    

    resi = [
        pickle.load(
            open(
                loc + file_list[di], 'rb'))
               for di in odd_idx
    ]
        
    return frac, resi


def pixel_timeline_masking_alpha_plot(frac, resi, block_id, pix_temperature):
    '''
    Pixel timeline masking plot for the temperature alpha 
    '''  
    ls = ['o', '*', 'x', 'o', '*', 'x']
    fig, axs = plt.subplots(figsize=(10, 8), nrows=2, ncols=1, sharey=False)
    fig.suptitle(r'Block: '+block_id+' Pixel Thermal mask')

    
    ax=axs[0]
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xticks(range(1, 21 + 1))
    ax.set_ylabel('Amplitude')

    textstr = '\n'.join((
        r'$\sigma_D=C_1$', ),)
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax.text(0.06, 0.95, textstr, transform=ax.transAxes, fontsize=18,
            verticalalignment='top', bbox=props)

    for pti in range(len(frac)):
        ax.plot(
            np.arange(1,22), frac[pti]['best-fit'], 
            ls[pti], label=r'Temperature: ' + pix_temperature[pti] + ' K')
    ax.axhline(0, color='black', linestyle='--')
    ax.legend(ncol=2)

    
    ax=axs[1]
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xticks(range(1, 21 + 1))
    ax.set_xlabel(r'Signal Position $[\alpha_{i}]$')
    ax.set_ylabel('Amplitude')

    textstr = '\n'.join((
        r'$\sigma_D=C_2$', ),)
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax.text(0.06, 0.95, textstr, transform=ax.transAxes, fontsize=18,
            verticalalignment='top', bbox=props)

    for pti in range(len(resi)):
        ax.plot(
            np.arange(1,22), resi[pti]['best-fit'], 
            ls[pti], label=r'Temperature: ' + pix_temperature[pti] + ' K')        
    ax.axhline(0, color='black', linestyle='--')
    
    
    fig.tight_layout()
    if savegig==True:
        fig.savefig('./pix_timeline_fitting_alpha.pdf')  
    fig.show()
    

    
def pixel_timeline_masking_1d_simulation(frac, resi, block_id, pix_temperature):
    '''
    Function for determining the 1d comparison between the simulation and the model
    '''
    fig, axs = plt.subplots(figsize=(20, 3*len(pix_temperature)), nrows=len(pix_temperature), ncols=2, sharey=False)

    fig.suptitle('Block:'+block_id+' Pixel; Timeline masking')
    for ri in range(len(pix_temperature)):
        ax=axs[ri,0]
        ax.plot(frac[ri][3], np.mean(frac[ri][0], axis=0), label='Observation')
        ax.plot(frac[ri][3], np.mean(frac[ri][1], axis=0), '--', label='Simulation')
        textstr = '\n'.join(
            (
            r'$\sigma_{D}=C_1$',
            r'Px Thermal=' + pix_temperature[ri] + ' K',
            r'FoM$=%.5f$' % (np.round(frac[ri][2], 5), ),)
        )

        props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
        ax.text(0.015, 0.95, textstr, transform=ax.transAxes, fontsize=18,
                verticalalignment='top', bbox=props)
        if ri==2:
            ax.set_xlabel('Frequency [MHz]')
        ax.set_ylabel('Temperature [K]')
        if ri==0:
            ax.legend()
         
        
        
        ax=axs[ri,1]
        ax.plot(resi[ri][3], np.mean(resi[ri][0], axis=0), label='Observation')
        ax.plot(resi[ri][3], np.mean(resi[ri][1], axis=0), '--', label='Simulation')
        textstr = '\n'.join(
            (
            r'$\sigma_{D}=C_2$',
            r'Px Thermal=' + pix_temperature[ri] + ' K',
            r'FoM$=%.2f$' % (np.round(resi[ri][2], 2), ),)
        )

        props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
        ax.text(0.015, 0.95, textstr, transform=ax.transAxes, fontsize=18,
                verticalalignment='top', bbox=props)
        if ri==2:
            ax.set_xlabel('Frequency [MHz]')

    fig.tight_layout()
    if savegig==True:
        fig.savefig('./pix_timeline_fitting.pdf')  
    fig.show()
    
    
def pixel_timeline_masking_2d_simulation(frac, resi, block_id, pix_temperature):
    '''
    Function for plotting the TOD plots
    '''
    
    plots_title = ['Observation', r'$C_1$', r'$C_2$']
    fig, axs = plt.subplots(figsize=(16, 3*len(pix_temperature)), nrows=len(pix_temperature), ncols=3, sharey=True, sharex=True)
    fig.suptitle('Block:'+block_id+' Thermal masking')
    
    for ri in range(len(pix_temperature)):
        for ci in range(3):
                xi = 3*ri+ci
                ax = axs[ri, ci]
                if ri==0:
                    ax.set_title(plots_title[ci])

                if ci==0:
                    cax = ax.imshow(
                        resi[ri][0], 
                        aspect='auto', 
                        extent=[
                            resi[ri][3][0], 
                            resi[ri][3][-1], 
                            pm.nd_s0[-1], 
                            pm.nd_s0[0]
                        ], 
                        vmax=np.max(mpt_sr[ri][0]), 
                        vmin=np.min(mpt_sr[ri][0])
                    )
                    
                elif ci==1:
                    cax = ax.imshow(
                        resi[ri][1], 
                        aspect='auto', 
                        extent=[
                            resi[ri][3][0], 
                            resi[ri][3][-1], 
                            pm.nd_s0[-1], 
                            pm.nd_s0[0]
                        ], 
                        vmax = np.max(mpt_sr[ri][0]), 
                        vmin=np.min(mpt_sr[ri][0])
                    )
                else:
                    cax = ax.imshow(
                        frac[ri][1], 
                        aspect = 'auto', 
                        extent = [
                            resi[ri][3][0], 
                            resi[ri][3][-1], 
                            pm.nd_s0[-1], 
                            pm.nd_s0[0]
                        ], 
                        vmax = np.max(mpt_sr[ri][0]), 
                        vmin = np.min(mpt_sr[ri][0])
                    )  

                cbar = fig.colorbar(cax, ax=ax)
                cbar.set_label(r'Temperature [K]', rotation=270, labelpad=20, y=0.45)

                if ci == 0:
                    ax.set_ylabel('Time [sec]')
                if ri == 2:
                    ax.set_xlabel('Frequency [MHz]')

    fig.tight_layout()
    if savegig == True:
        fig.savefig('./pix_timeline_fitting_waterfall.pdf')


In [ ]:
pixel_temperature = ['200', '250', '300', '350']
# pixel_temperature = ['200']

pt_mask_dic = {}
for bi in range(len(blocks)):
    
    # Alpha values
    frac, resi = pixel_timeline_alpha(block_id=blocks[bi], folder_id=folders[bi])
    
    pt_mask_dic.update({blocks[bi] : [frac, resi]})
    
    pixel_timeline_masking_alpha_plot(frac=frac, resi=resi, 
                                      block_id=blocks[bi], pix_temperature=pixel_temperature)

    # Simulation
    mpt_sf = [chisq_func2(
        block_id = blocks[bi], 
        a_param = frac[pti]['best-fit'], 
        s_param = None, 
        damper = None,                  
        frequency_slice = [1100, 1350], 
        time_slice = [None, None], 
        t_mask = False, 
        d_mask = False, 
        pt_mask = int(ptemp),
        time_avg = False, 
        chi_sigma = True, 
        verbose = False
    ) 
             for pti, ptemp in enumerate(pixel_temperature)]
  
    mpt_sr = [chisq_func2(
        block_id = blocks[bi], 
        a_param = resi[pti]['best-fit'], 
        s_param = None, 
        damper = None, 
        frequency_slice = [
            1100, 
            1350
        ], 
        time_slice = [
            None, 
            None
        ], 
        t_mask = False,
        d_mask = False,
        pt_mask = int(ptemp),
        time_avg = False, 
        chi_sigma = False, 
        verbose = False
    ) 
             for pti, ptemp in enumerate(pixel_temperature)]
    
    
    
    pixel_timeline_masking_1d_simulation(
        frac = mpt_sf,
        resi = mpt_sr, 
        block_id = blocks[bi],
        pix_temperature = pixel_temperature
    )
    
    pixel_timeline_masking_2d_simulation(
        frac = mpt_sf,
        resi = mpt_sr, 
        block_id = blocks[bi],
        pix_temperature = pixel_temperature
    )

### Temporal-masking

In [ ]:
def temporal_masking_alpha(block_id, folder_id):
    loc = '/idia/projects/hi_im/satellite_rfi/Testing/'+block_id+'/'+folder_id+'/time_zones/'
    
    if block_id == str(1551055211):
        ts = [775, 2200, 5500]
        te = [1000, 2400, 6200]        
        
    elif block_id == str(1562857793):
        ts = [986, 1868, 2338, 4815]
        te = [1850, 2257, 3223, 6689]
        
    elif block_id == str(1556138397):
        ts = [936, 2183]
        te = [1296, 6634]
        
    elif block_id == str(1556052116):
        ts = [1102, 1724, 5404]
        te = [1225, 4462, 6673]
        
    elif block_id == str(1554156377):
        ts=[938, 2617]
        te=[1365, 6641]
        
    elif block_id == str(1553966342):
        ts=[1005.5]
        te=[4505]
        
    elif block_id == str(1551037708):
        ts=[737.175, 1762.75, 4625.55]
        te=[1756, 3543.9, 6440.7]
        
        # ts=[851, 3543]
        # te=[1757, 4623.55]
    else:
        print ('Check block')
    
    time_zone = [str(ts[ti]) + '-' + str(te[ti]) for ti in range(len(ts))]

    file_list = os.listdir(loc)
    file_list.sort()

    file_list_s = [] 
    for tsi in ts:
        for fl in file_list:
            if str(tsi) in fl:
                file_list_s.append(fl)

    idx = np.arange(len(file_list_s))
    even_idx = idx[::2]
    odd_idx = idx[1::2]

    frac = [pickle.load(
        open(
            loc+file_list_s[di],'rb'
        )
    )
               for di in even_idx]

    resi = [pickle.load(
        open(
            loc+file_list_s[di],'rb'
        )
    )
               for di in odd_idx]
    
    return frac, resi, time_zone, ts, te

def temporal_masking_alpha_plot(frac, resi, time_start, time_end, time_zone, block_id):
    '''
    Function for plotting the alpha values for the temporal masking scenario
    '''
    ls = ['o', 'x']
    fig, axs = plt.subplots(figsize=(10, len(time_start)*4), nrows=len(time_start), ncols=1, sharex=True, sharey=True)
    fig.suptitle(r'Block: '+block_id+' Temporal Alpha')

    for ri in range(len(time_start)):
        if len(time_start)>1:
            ax=axs[ri]
        else:
            ax=axs
        textstr = '\n'.join((
            r'Time zone: '+str(time_zone[ri])+' sec', ),)

        props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
        # place a text box in upper left in axes coords
        ax.text(0.06, 0.95, textstr, transform=ax.transAxes, fontsize=18,
                verticalalignment='top', bbox=props)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_xticks(range(1, 21 + 1))
        ax.set_ylabel('Amplitude')
        ax.plot(np.arange(1,22), frac[ri]['best-fit'], ls[0], alpha=0.6, label=r'$\chi_{\sigma}$=$C_2$')
        ax.plot(np.arange(1,22), resi[ri]['best-fit'], ls[1], label=r'$\chi_{\sigma}=C_1$')
        ax.axhline(0, linestyle='--', color='k')
        if ri==len(ts):
            ax.set_xlabel(r'Signal Position $[\alpha_{i}]$')
        if ri==0:
            ax.legend(loc='best')

    fig.tight_layout()
    if savegig==True:
        fig.savefig('./temporal_fitting_alpha.pdf')

    fig.show()
    
def temporal_masking_1d_simulation(frac, resi, time_start, time_end, time_zone, block_id):
    fig, axs = plt.subplots(figsize=(20, 3*len(ts)), nrows=len(time_start), ncols=2, sharey=True)
    fig.suptitle(r'Block: '+block_id+' Temporal Masking')

    for ri in range(len(time_start)):
        if len(time_start)>1:
            ax=axs[ri,0]
        else:
            ax=axs[0]
        ax.plot(frac[ri][3], np.ma.mean(frac[ri][0], axis=0), label='Observation')
        ax.plot(frac[ri][3], np.ma.mean(frac[ri][1], axis=0), '--', label='Simulation')
        textstr = '\n'.join((
            r'$\sigma_D=C_1$',
            r'Time='+time_zone[ri]+' sec',
            r'FoM$=%.5f$' % (np.round(frac[ri][2], 5), ),))

        props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
        ax.text(0.015, 0.95, textstr, transform=ax.transAxes, fontsize=18,
                verticalalignment='top', bbox=props)

        if ri==len(ts):
            ax.set_xlabel('Frequency [MHz]')
        ax.set_ylabel('Temperature [K]')
        if ri==0:
            ax.legend()

        if len(time_start)>1:
            ax=axs[ri,1]
        else:
            ax=axs[1]
        ax.plot(resi[ri][3], np.ma.mean(resi[ri][0], axis=0), label='Observation')
        ax.plot(resi[ri][3], np.ma.mean(resi[ri][1], axis=0), '--', label='Simulation')
        textstr = '\n'.join((
            r'$\sigma_D=C_2$',
            r'Time='+time_zone[ri]+' sec',
            r'FoM$=%.2f$' % (np.round(resi[ri][2], 2), ),))

        props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
        ax.text(0.015, 0.95, textstr, transform=ax.transAxes, fontsize=18,
                verticalalignment='top', bbox=props)

        if ri==len(ts):
            ax.set_xlabel('Frequency [MHz]')

    fig.tight_layout()
    if savegig==True:
        fig.savefig('./temporal_fitting.pdf')

    fig.show()
    
def temporal_masking_2d_simulation(frac, resi, time_start, time_end, time_zone, block_id):
    plots_title = ['Observation', r'$C_1$', r'$C_2$']
    
    fig, axs = plt.subplots(
        figsize=(16, 3*len(time_start)), 
        nrows=len(time_start), 
        ncols=3, 
        sharey=False, 
        sharex=True
    )
    
    fig.suptitle(r'Block: ' + block_id + ' Temporal Masking')
    for ri in range(len(time_start)):
        for ci in range(3):
            xi = 3*ri+ci
            if len(time_start)>1:
                ax = axs[ri, ci]
            else:
                ax = axs[ci]
            if ri == 0:
                ax.set_title(plots_title[ci])
            if ci == 0:
                cax = ax.imshow(
                    resi[ri][0], 
                    aspect='auto', 
                    extent=[
                        resi[ri][3][0], 
                        resi[ri][3][-1], 
                        time_end[ri], 
                        time_start[ri]
                    ], 
                    vmax=np.max(mtz_sr[ri][0]), 
                    vmin=np.min(mtz_sr[ri][0])
                )
                
            elif ci == 1:
                cax = ax.imshow(
                    frac[ri][1], 
                    aspect='auto', 
                    extent=[
                        resi[ri][3][0], 
                        resi[ri][3][-1], 
                        time_end[ri], 
                        time_start[ri]
                    ], 
                    vmax=np.max(mtz_sr[ri][0]), 
                    vmin=np.min(mtz_sr[ri][0])
                )
                
            else:
                cax = ax.imshow(
                    resi[ri][1], 
                    aspect='auto', 
                    extent=[
                        resi[ri][3][0], 
                        resi[ri][3][-1], 
                        time_end[ri], 
                        time_start[ri]
                    ], 
                    vmax=np.max(mtz_sr[ri][0]), 
                    vmin=np.min(mtz_sr[ri][0])
                )
                
            cbar = fig.colorbar(cax, ax=ax)
            cbar.set_label(r'Temperature [K]', rotation=270, labelpad=20, y=0.45)

            if ci == 0:
                ax.set_ylabel('Time [sec]')
            if ri == 2:
                ax.set_xlabel('Frequency [MHz]')

    fig.tight_layout()
    if savegig == True:
        fig.savefig('./temporal_fitting_waterfall.pdf')
        
    fig.show()

In [ ]:
temporal_masking_dic = {}
for bi in range(len(blocks)):
    # Alpha parameters
    frac, resi, time_zone, ts, te = temporal_masking_alpha(
        block_id = blocks[bi], 
        folder_id = folders[bi]
    )
    
    temporal_masking_dic.update({blocks[bi] : [frac, resi]})
    temporal_masking_alpha_plot(
        frac = frac, 
        resi = resi, 
        time_start = ts, 
        time_end = te, 
        time_zone = time_zone, 
        block_id = blocks[bi]
    )
    
    # Simulation
    
    mtz_sf = [chisq_func2(
        block_id = blocks[bi],
        a_param = frac[tz]['best-fit'],
        s_param = None,
        damper = None,
        frequency_slice = [
            1100, 
            1350
        ], 
        time_slice = [
            ts[tz],
            te[tz]
        ], 
        t_mask = False,
        d_mask = False,
        time_avg = False, 
        chi_sigma = True, 
        verbose = False
    ) for tz,time in enumerate(time_zone)
             ]

    mtz_sr = [chisq_func2(
        block_id = blocks[bi],
        a_param = resi[tz]['best-fit'],
        s_param = None,
        damper = None,
        frequency_slice = [
            1100, 
            1350
        ], 
        time_slice = [
            ts[tz],
            te[tz]
        ], 
        t_mask = False,
        d_mask = False,
        time_avg = False, 
        chi_sigma = False,
        verbose = False
    ) for tz,time in enumerate(time_zone)
             ]
    
    
    temporal_masking_1d_simulation(
        frac = mtz_sf,
        resi = mtz_sr,
        time_start = ts,
        time_end = te,
        time_zone = time_zone,
        block_id = blocks[bi]
    )
    
    temporal_masking_2d_simulation(
        frac = mtz_sf,
        resi = mtz_sr,
        time_start = ts,
        time_end = te,
        time_zone = time_zone,
        block_id = blocks[bi]
    )
    

### Temporal-masking + Coarse grain

In [ ]:
def temporal_masking_coarse_alpha(block_id, folder_id):
    loc = '/idia/projects/hi_im/satellite_rfi/Testing/'+block_id+'/'+folder_id+'/time_average/'
    
    if block_id==str(1551055211):
        ts = [775, 2200, 5500]
        te = [1000, 2400, 6200]        
    elif block_id==str(1562857793):
        ts = [986, 1868, 2338, 4815]
        te = [1850, 2257, 3223, 6689]    
    elif block_id==str(1556138397):
        ts = [936, 2183]
        te = [1296, 6634]
    elif block_id==str(1556052116):
        ts = [1102, 1724, 5404]
        te = [1225, 4462, 6673]
    elif block_id==str(1554156377):
        ts=[938, 2617]
        te=[1365, 6641]
    elif block_id==str(1553966342):
        ts=[1005.5]
        te=[4505]
    elif block_id==str(1551037708):
        ts=[737.175, 1762.75, 4625.55]
        te=[1756, 3543.9, 6440.7]

    else:
        print ('Check block')
    
    time_zone = [str(ts[ti])+'-'+str(te[ti]) for ti in range(len(ts))]

    file_list = os.listdir(loc)
    file_list.sort()

    file_list_s = [] 
    for tsi in ts:
        for fl in file_list:
            if str(tsi) in fl:
                file_list_s.append(fl)

    idx = np.arange(len(file_list_s))
    even_idx = idx[::2]
    odd_idx = idx[1::2]

    frac = [pickle.load(open(loc+file_list_s[di],'rb'))
               for di in even_idx]

    resi = [pickle.load(open(loc+file_list_s[di],'rb'))
               for di in odd_idx]
    
    return frac, resi, time_zone, ts, te


def temporal_masking_coarse_1d_simulation(frac, time_start, time_end, time_zone, block_id):
    fig, axs = plt.subplots(figsize=(20, 4*len(time_start)), nrows=len(time_start), ncols=1, sharey=True)

    fig.suptitle(r'Block: '+block_id+' Temporal masking & coarse grain')
    for ri in range(len(time_start)):
        if len(time_start)>1:
            ax=axs[ri]
        else:
            ax=axs
        ax.plot(frac[ri][3], np.ma.mean(frac[ri][0], axis=0), label='Observation')
        ax.plot(frac[ri][3], np.ma.mean(frac[ri][1], axis=0), '--', label='Simulation')
        textstr = '\n'.join((
            r'$\sigma_D=C_1$',
            r'Time='+time_zone[ri]+' sec',
            r'$\chi^2/N=%.5f$' % (np.round(frac[ri][2], 5), ),))

        props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
        ax.text(0.015, 0.95, textstr, transform=ax.transAxes, fontsize=18,
                verticalalignment='top', bbox=props)

        if ri==len(ts):
            ax.set_xlabel('Frequency [MHz]')
        ax.set_ylabel('Temperature [K]')
        ax.legend()
        fig.tight_layout()
        # fig.savefig('./temporal2_fitting.pdf')

        fig.show()
        
def temporal_masking_coarse_2d_simulation(frac, time_start, time_end, time_zone, block_id):
    plots_title = ['Observation', r'$C_1$']
    fig, axs = plt.subplots(figsize=(16, 3*len(time_start)), nrows=len(time_start), ncols=2, sharex=True)
    fig.suptitle(r'Block: '+block_id+' Temporal masking & coarse grain')
    for ri in range(len(time_start)):
        for ci in range(2):
            xi = 3*ri+ci
            if len(time_start)>1:
                ax = axs[ri, ci]
            else:
                ax = axs[ci]
            if ri==0:
                ax.set_title(plots_title[ci])
            if ci==0:
                cax = ax.imshow(mtza_sf[ri][0], aspect='auto', extent=[f_slice[0], f_slice[-1], te[ri], ts[ri]], vmax=np.max(mtz_sf[ri][0]), vmin=np.min(mtz_sf[ri][0]))
            elif ci==1:
                cax = ax.imshow(mtza_sf[ri][1], aspect='auto', extent=[f_slice[0], f_slice[-1], te[ri], ts[ri]], vmax=np.max(mtz_sr[ri][0]), vmin=np.min(mtz_sr[ri][0]))  

            cbar = fig.colorbar(cax, ax=ax)
            cbar.set_label(r'Temperature [K]', rotation=270, labelpad=20, y=0.45)

            if ci==0:
                ax.set_ylabel('Time [sec]')
            if ri==2:
                ax.set_xlabel('Frequency [MHz]')

    fig.tight_layout()
    # fig.savefig('./temporal2_fitting_waterfall.pdf')
    
    fig.show()


In [ ]:
temporal_masking_coarse_dic = {}
for bi in range(len(blocks)):
    # Alpha parameters
    frac, resi, time_zone, ts, te = temporal_masking_coarse_alpha(block_id=blocks[bi], folder_id=folders[bi])
    temporal_masking_coarse_dic.update({blocks[bi] : [frac, resi]})
    temporal_masking_alpha_plot(frac=frac, resi=resi, time_start=ts, time_end=te, time_zone=time_zone, block_id=blocks[bi])
    
    mtza_sf = [chisq_func2(block_id=blocks[bi],a_param=frac[tz]['best-fit'], s_param=None, damper=None, frequency_slice=[1100, 1350], time_slice=[ts[tz], te[tz]], 
                     t_mask=False, d_mask=False ,time_avg=10, chi_sigma=True, verbose=False) for tz,time in enumerate(time_zone)]
    
    temporal_masking_coarse_1d_simulation(frac=mtza_sf, time_start=ts, time_end=te, time_zone=time_zone, block_id=blocks[bi])
    temporal_masking_coarse_2d_simulation(frac=mtza_sf, time_start=ts, time_end=te, time_zone=time_zone, block_id=blocks[bi])

In [ ]:
stop

## Running for the extended frequency range

Code for running the data without the background model

Note that the 2d angular satellite maps should be in the correct frequency range.

Note that the frequency range should be further than the length set

Once you set the number of satellites you cannot increase or decrease the satellite information. Have to keep the same length of satellites

In [ ]:
%load_ext autoreload
%autoreload 2

%reload_ext autoreload

In [ ]:
def chisq_func3(block_id, a_param, s_param=None, damper=None, satellites_only=True, 
                sat_info=pm.satellite_catalogue, add_sub=[False, True],
                frequency_slice=False, time_slice=False, t_mask=False, d_mask=False, time_avg=False, verbose=False):
    
    data_save='/idia/projects/hi_im/satellite_rfi/Testing/'+block_id+'/'
    # Needed this function to update the data information
    nd_s0, nd_s0_coords, nd_s0_coords2, nd_s0_pos, frequency = katdal_information(data_save=data_save+block_id)
    
    if d_mask==False:
        satellites=None
    else:
        satellites=pm.data_save+'nearby_satellites/nearby_satellite_close_angle_'+d_mask+'.p'
    
    """INITIALIZING THE SATELLITE FUNCTION"""
    sat = ss(file_name=block_id, 
                 sats_only=satellites_only, 
                 data_loc=data_save, 
                 sat_loc=data_save,
                 survey_info=[nd_s0, nd_s0_coords, frequency], 
                 sat_info=sat_info,
                 plots_loc=data_save,
                 sat_beam=pm.beam_model+'_beam_'+str(950)+'_'+str(1500)+'MHz', 
                 frequency_range=[950, 1500], 
                 constellations=pm.constellations_remain,
                 nearby_satellites=satellites,
                 verbose=False)
        
    """EXCECUTING THE THE SATELLITE SIM FUNCTION"""
    sat.excecute(a_param=a_param, 
                 obs_time_start=time_slice[0], obs_time_end=time_slice[1], 
                 obs_frequency_start=frequency_slice[0], obs_frequency_end=frequency_slice[1], 
                 file_bias_choice=pm.bias, 
                 add_sub=[add_sub[0], add_sub[1]], 
                 attenuation_func=damper, 
                 attenuation_sigma=s_param, 
                 bandsize=None,
                 verbose=False)

    """FREQUENCY SLICE"""
    frange_slice = sat.frequency_band[sat.frequency_idx[0]:sat.frequency_idx[1]]  
    if satellites_only==False:
        """MASKING"""
        ## TEMPERATURE MASKING 
        if t_mask!=False and d_mask==False:
            print ('Temperature mask of '+str(t_mask)+' Kelvin')
            zero_arr = np.zeros(sat.calibration_data_slice.shape)
            mask_idx = np.where(sat.calibration_data_slice > t_mask)

            zero_arr[mask_idx]=1
            simulation = np.ma.array(data=sat.simulation_TOD_slice.T, mask=zero_arr.T)    #SIMULATIONS
            data = np.ma.array(data=sat.calibration_data_slice.T, mask=zero_arr.T)  #DATA

        ## DEGREE MASKING
        elif d_mask!=False and t_mask==False:
            print ("Area mask of "+str(d_mask)+" degrees")
            simulation = np.ma.array(data=sat.simulation_TOD_slice.T, mask=sat.mask_nearby_satellites_slice)
            data = np.ma.array(data=sat.calibration_data_slice.T, mask=sat.mask_nearby_satellites_slice)

        ## NO MASKING
        else:
            print ("No masking applied")
            simulation = np.ma.array(data=sat.simulation_TOD_slice.T, mask=None)
            data = np.ma.array(data=sat.calibration_data_slice.T, mask=None)
        return data, simulation, frange_slice#, sat.sat_data_adjusted
    else:
        if d_mask!=False:
            print ('Masking and Satellites Only')
            simulation = np.ma.array(data=sat.simulation_TOD_slice.T, mask=sat.mask_nearby_satellites_slice)
        else:
            print ("No masking and Satellites Only")
            simulation = np.ma.array(data=sat.simulation_TOD_slice.T, mask=None)
        return simulation, frange_slice, nd_s0, nd_s0_pos#, sat.sat_data_adjusted        
        


    

In [ ]:
def residual_level4_maps(block_id, antenna_no):
    '''
    Function to obtain the resiuald map from the data set that Mel gave.
    The data is tored at the same location 
    Parameters: 
        block_id - the block/file number
        antanna_no - the antenna number is in use
    '''
    
    hh4 = pickle.load(open('/users/bengelbrecht/Tresid_'+block_id+'_'+antenna_no+'h_l4masks', 'rb'), encoding='latin1')
    vv4 = pickle.load(open('/users/bengelbrecht/Tresid_'+block_id+'_'+antenna_no+'v_l4masks', 'rb'), encoding='latin1')
    
    return hh4, vv4

# Obtaining the frequecny index information for the start and end. This is based on the fact that eah block as the same frequency information
fs, fe = tools.find_idx(data_array=pm.frequency, data_variable=973), tools.find_idx(data_array=pm.frequency, data_variable=1015.2)

def extended_cosmo_band_2d(block_id, data_info, data_info_name, frequency, time):
    fig, axs = plt.subplots(figsize=(6*len(data_info),6), nrows=1, ncols=len(data_info))
    fig.suptitle('Block: '+block_id+' Data vs Model comparison')
    
    for di in range(len(data_info)):
        ax=axs[di]
        ax.set_title(data_info_name[di])
        ax.set_title(data_info_name[di])
        cax = ax.imshow((data_info[di]), aspect='auto', extent=[frequency[0], frequency[1], time[0], time[-1]]) 
        cbar = fig.colorbar(cax, ax=ax)
        cbar.set_label(r'Temperature [K]', rotation=270, labelpad=20, y=0.45)
        ax.set_xlabel('Frequency [MHz]')
        ax.set_ylabel('Time [sec]')
        
    fig.tight_layout()
    fig.show()
    

def extended_cosmo_band_1d_plot(block_id, flist, data_info, data_info_name, time):
    '''
    Function for frequency response in the cosmological band, only six frequency indices should be selected
    '''
    fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(16, 8), sharey=True, sharex=True)
    fig.suptitle('Block: '+block_id+' 1D frequency comparison')
    for ri in range(2):
        for ci in range(3):
            x = 3*ri+ci
            ax = axs[ri,ci]
            
            for di in range(len(data_info)):
                ax.plot(time, data_info[di][:, flist[x]], alpha=1-0.3*di, label=data_info_name[di])

                
            if ci==0:
                ax.set_ylabel('Temperature [K]')
            if ri==1:
                ax.set_xlabel('Time [sec]')

            textstr = '\n'.join((
                r'$\nu$: '+str(frange_reduced[flist[x]])+' MHz',))

            props = dict(boxstyle='round', facecolor='wheat', alpha=0.2)
            # place a text box in upper left in axes coords
            ax.text(0.02, 0.95, textstr, transform=ax.transAxes, fontsize=14,
                    verticalalignment='top', bbox=props)

            if x==4:
                ax.legend(ncol=3, frameon=True, loc='lower center', fontsize=14)
            

    fig.tight_layout()
    if savegig==True:
        fig.savefig('./cosmological_band.pdf')
    fig.show()
    
def rmsValue(arr):
    '''
    Function that produces the root mean sqaure of an array
    Parameters: 
        arr - array of values
    Return 
        RMS
    '''
    # Mean of the array
    arr_mean = np.ma.mean(arr)
    # getting the length of the array
    n = float(len(arr))
    #Calculate square
    square = np.ma.sum((arr-arr_mean)**2) /  float(n)
    #Calculate Root
    root = np.ma.sqrt(square)
     
    return root

def rms_values_cosmo_band(block_id, frequency, hhpol, vvpol, model):
    fig, axs = plt.subplots(figsize=(16, 4), nrows=1, ncols=2, sharey=True)
    fig.suptitle('Block: '+block_id+' RMS calculation')
    ax=axs[0]
    ax.plot(frequency, hhpol[1], '.', label='Flag:Lvl 4')
    ax.plot(frequency, hhpol[2], '.', label='Flag:Lvl 4 + 5F ')
    ax.plot(frequency, model[0], '.', label='Sim:Lvl 4 + 5F ')
    ax.legend(loc='upper right', fontsize=15)
    ax.set_xlabel('Frequency [MHz]')
    ax.set_ylabel('RMS')
    ax.set_title('RMS for HH-pol')
    
    ax=axs[1]
    ax.plot(frequency, vvpol[1], '.', label='Flag:Lvl 4')
    ax.plot(frequency, vvpol[2], '.', label='Flag:Lvl 4 + 5F ')
    ax.plot(frequency, model[0], '.', label='Sim:Lvl 4 + 5F ')
    ax.legend(loc='upper right', fontsize=15)
    ax.set_xlabel('Frequency [MHz]')
    ax.set_title('RMS for VV-pol')    
    fig.tight_layout()
    # if savegig==True:
        # fig.savefig('./temporal_fitting_residual_ang.pdf')
    fig.show()
    
def rms_values_cosmo_band_total(block_id, frequency, pol, model):
    fig, axs = plt.subplots(figsize=(16, 4), nrows=1, ncols=1, sharey=True)
    ax=axs
    ax.plot(frequency, pol[1], '.', label='Flag:Lvl 4')
    ax.plot(frequency, pol[2], '.', label='Flag:Lvl 4 + 5F ')
    ax.plot(frequency, model[0], '.', label='Sim:Lvl 4 + 5F ')
    ax.legend(loc='upper right', fontsize=15)
    ax.set_xlabel('Frequency [MHz]')
    ax.set_ylabel('RMS')
    ax.set_title('Block: '+block_id+' RMS for (HH+VV)/2')

    # if savegig==True:
        # fig.savefig('./temporal_fitting_residual_ang.pdf')
    fig.show()


In [ ]:
alpha_choice = [0,0,0,0,1,1,0]
degree_masking_type = ['3F', '5F', '5F', '5F', '5F', '5F', '5F']
thermal_masking_type = [False, False, False, False, False, False, False]

In [ ]:
rms_hh_val, rms_vv_val, rms_hhvv_val, rms_model_val = [], [], [], []

for ei in range(len(blocks)):
    # Note the codes needs the frequency range to extend beyond the central frequencies within the data
    #     sim_ext, frange_ext, nd_s0, nd_s0_pos = chisq_func3(block_id=blocks[ei], a_param=frac[1]['best-fit'], s_param=None, damper=None, 
    #                                       frequency_slice=[973, 1350], time_slice=[None, None], t_mask=50, d_mask=False ,time_avg=False, verbose=False)
    
    sim_ext, frange_ext, nd_s0, nd_s0_pos = chisq_func3(block_id=blocks[ei], 
                                                        a_param=temporal_masking_dic[blocks[ei]][0][alpha_choice[ei]]['best-fit'],
                                                        s_param=None, damper=None,frequency_slice=[973, 1350], time_slice=[None, None], 
                                                        t_mask=thermal_masking_type[ei], d_mask=degree_masking_type[ei],
                                                        time_avg=False, verbose=False)


    # Obtaining the residual information for the level 4 information of the various blocks
    if blocks[ei]=='1551055211' or blocks[ei]=='1556052116' or blocks[ei]=='1556138397' or\
    blocks[ei]=='1551037708' or blocks[ei]=='1553966342' or blocks[ei]=='1554156377':
        ant_no='m000'
    if blocks[ei]=='1562857793':
        ant_no='m004'

    print (blocks[ei])
    print (ant_no)
    print (alpha_choice[ei])
    print (degree_masking_type[ei])
    print (thermal_masking_type[ei])   

    resi_hh4, resi_vv4 = residual_level4_maps(block_id=blocks[ei], antenna_no=ant_no)
    resi_hhvv4 = (resi_hh4+resi_vv4)/2

    # Obtaining the reduced frequency range to match the cosmological banf inforation
    fend_idx = np.where(frange_ext>1015)[0][0]
    frange_reduced = frange_ext[:fend_idx]
    sim_e_reduced = sim_ext[:, :fend_idx]

    # Reducing the size of the level 4 residual and masking it similar to that of the extended simulation 
    resi_hh4_2 = resi_hh4[nd_s0_pos, fs:fe][:-1, :]
    resi_hh4_2m = np.ma.array(data=resi_hh4_2, mask=sim_e_reduced.mask)

    resi_vv4_2 = resi_vv4[nd_s0_pos, fs:fe][:-1, :]
    resi_vv4_2m = np.ma.array(data=resi_vv4_2, mask=sim_e_reduced.mask)

    resi_hhvv4_2 = resi_hhvv4[nd_s0_pos, fs:fe][:-1, :]
    resi_hhvv4_2m = np.ma.array(data=resi_hhvv4_2, mask=sim_e_reduced.mask)

    # Masking the simulation with respect the level 4 residuals
    sim_em_reduced = np.ma.array(data=sim_e_reduced, mask=resi_hh4_2m.mask)

    # Must check what it do
    resi_hh4_2mt = np.zeros((resi_hh4_2.shape))
    resi_hh4_2mt = np.ma.array(data=resi_hh4_2mt, mask=resi_hh4_2m.mask)

    resi_vv4_2mt = np.zeros((resi_vv4_2.shape))
    resi_vv4_2mt = np.ma.array(data=resi_vv4_2mt, mask=resi_vv4_2m.mask)

    resi_hhvv4_2mt = np.zeros((resi_hhvv4_2.shape))
    resi_hhvv4_2mt = np.ma.array(data=resi_hhvv4_2mt, mask=resi_hhvv4_2m.mask)

    sim_emt_reduced = np.zeros((sim_em_reduced.shape))
    sim_emt_reduced = np.ma.array(data=sim_emt_reduced, mask=sim_em_reduced.mask)

    # Subtracting the mean per frequency channel
    for i in range(resi_hh4_2.shape[1]):
        resi_hh4_2mt[:, i] = resi_hh4_2m[:, i] - np.ma.mean(resi_hh4_2m[:, i])
        resi_vv4_2mt[:, i] = resi_vv4_2m[:, i] - np.ma.mean(resi_vv4_2m[:, i])
        resi_hhvv4_2mt[:, i] = resi_hhvv4_2m[:, i] - np.ma.mean(resi_hhvv4_2m[:, i])
        sim_emt_reduced[:, i] = sim_em_reduced[:, i] - np.ma.mean(sim_em_reduced[:, i])

    data_info = [sim_emt_reduced, resi_hhvv4_2mt, resi_hh4_2mt, resi_vv4_2mt]
    data_info_name = ['Model', '(HH+VV)/2', 'HH', 'VV']
    # Plotting
    extended_cosmo_band_2d(block_id=blocks[ei], data_info=data_info, data_info_name=data_info_name,
                           frequency=[frange_reduced[0], frange_reduced[-1]], time=[nd_s0[-1], nd_s0[0]])

    # List of different frequency index to select from
    flist = [0, 25, 56, 75, 91, 110]
    # Range of frequency index to chose from
    clen=15

    # Plotting
    extended_cosmo_band_1d_plot(block_id=blocks[ei], flist=flist, data_info=data_info, data_info_name=data_info_name, time=nd_s0[:-1])

    # Calculating the RMS for the cosmology band frequencies
    hh = [resi_hh4_2.data, resi_hh4_2, resi_hh4_2mt]
    hh_val = []

    vv = [resi_vv4_2.data, resi_vv4_2, resi_vv4_2mt]
    vv_val = []

    hhvv = [resi_hhvv4_2.data, resi_hhvv4_2, resi_hhvv4_2mt]
    hhvv_val = []

    sims = [sim_emt_reduced]
    sims_val = []

    n = resi_hh4_2mt[:, 0].shape[0]
    nf = resi_hh4_2mt.shape[1]

    for resi_hh in hh:
        hh_val.append([rmsValue(arr=resi_hh[:, fi]) for fi in range(nf)])

    for resi_vv in vv:
        vv_val.append([rmsValue(arr=resi_vv[:, fi]) for fi in range(nf)])

    for resi_hhvv in hhvv:
        hhvv_val.append([rmsValue(arr=resi_hhvv[:, fi]) for fi in range(nf)])

    for resi_sims in sims:
        sims_val.append([rmsValue(arr=resi_sims[:, fi]) for fi in range(nf)])

    # Plotting
    rms_values_cosmo_band(block_id=blocks[ei], frequency=frange_reduced, hhpol=hh_val, vvpol=vv_val, model=sims_val)

    rms_values_cosmo_band_total(block_id=blocks[ei], frequency=frange_reduced, pol=hhvv_val, model=sims_val)

    rms_hh_val.append(hh_val)
    rms_vv_val.append(vv_val)
    rms_hhvv_val.append(hhvv_val)
    rms_model_val.append(sims_val)

    
rms_hh_val = np.array(rms_hh_val)  
rms_vv_val = np.array(rms_vv_val)  
rms_hhvv_val = np.array(rms_hhvv_val)  
rms_model_val = np.array(rms_model_val)  

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4), nrows=1, ncols=1)
for si in range(len(rms_model_val)):
    ax.plot(frange_reduced, rms_model_val[si, 0], label=blocks[si])

ax.set_xlabel('Frequency [MHz]')
ax.set_ylabel('RMS value')
ax.set_title('Model RMS')
ax.legend(ncol=3, fontsize=10)
fig.tight_layout()
    

In [ ]:
rms_hh_val[0, 1].shape

In [ ]:
fig, axs = plt.subplots(figsize=(16, 8), nrows=2, ncols=1, sharex=True)

ax=axs[0]
for si in range(len(rms_model_val)):
    ax.plot(frange_reduced, rms_hhvv_val[si, 1], label=blocks[si])

ax.set_ylabel('RMS value')
ax.set_title('Data RMS: Flag Lvl4')
ax.legend(ncol=3, fontsize=10)

ax=axs[1]
for si in range(len(rms_model_val)):
    ax.plot(frange_reduced, rms_hhvv_val[si, 2], label=blocks[si])

ax.set_ylabel('RMS value')
ax.set_title('Data RMS: Flag Lvl4 + 5F')
ax.legend(ncol=3, fontsize=10)
ax.set_xlabel('Frequency [MHz]')


fig.tight_layout()
    

In [ ]:
rms_model_val.shape

In [ ]:
fig, axs = plt.subplots(figsize=(16, 8), nrows=2, ncols=1, sharex=True)

ax=axs[0]
ax.plot(frange_reduced, np.ma.mean(rms_hhvv_val, axis=0)[1], label='7 Data Blocks')
ax.plot(frange_reduced, np.ma.mean(rms_model_val, axis=0)[0], label='7 Model Blocks')

ax.set_ylabel('RMS value')
ax.set_title('Data RMS: Flag Lvl4')
ax.legend(ncol=3, fontsize=10)

ax=axs[1]
ax.plot(frange_reduced, np.ma.mean(rms_hhvv_val, axis=0)[2], label='7 Data Blocks')
ax.plot(frange_reduced, np.ma.mean(rms_model_val, axis=0)[0], label='7 Model Blocks')

ax.set_ylabel('RMS value')
ax.set_title('Data RMS: Flag Lvl4 + 5F')
ax.legend(ncol=3, fontsize=10)
ax.set_xlabel('Frequency [MHz]')


fig.tight_layout()
    